<a href="https://colab.research.google.com/github/JulianNguyen05/xiangqi-sorting-jetmax/blob/main/training/train_yolo11.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Chuẩn bị môi trường
Kiểm tra GPU, mount Drive và cài ultralytics.

In [ ]:
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'KHÔNG CÓ GPU - đổi runtime sang T4')

from google.colab import drive
drive.mount('/content/drive')

!pip install -q ultralytics

In [ ]:
import os
BASE = '/content/drive/MyDrive/datn'
OLD_RUN = f'{BASE}/runs/yolo11n_640'          # lần train trước (degrees=10)
for p in [
    f'{BASE}/merged/merged_v1.zip',
    f'{OLD_RUN}/weights/best.pt',
]:
    print(os.path.exists(p), p)

## Giải nén dataset hợp nhất
Copy merged_v1.zip từ Drive về máy Colab (đọc nhanh hơn đọc trực tiếp từ Drive) và kiểm tra số ảnh/nhãn mỗi tập.

In [ ]:
import os, shutil

BASE = '/content/drive/MyDrive/datn'
MERGED = '/content/merged'

if not os.path.isdir(f'{MERGED}/train'):
    shutil.unpack_archive(f'{BASE}/merged/merged_v1.zip', MERGED)

for s in ['train', 'val', 'test']:
    print(s, len(os.listdir(f'{MERGED}/{s}/images')), 'ảnh |', len(os.listdir(f'{MERGED}/{s}/labels')), 'nhãn')

print(open(f'{MERGED}/data.yaml').read())

## Kiểm tra nhãn trước khi huấn luyện
Đảm bảo mỗi ảnh đều có file nhãn tương ứng và mọi class id nằm trong khoảng 0-13. Nhãn sai class id sẽ làm quá trình train báo lỗi hoặc cho kết quả sai.

In [ ]:
import glob

bad = 0
for s in ['train', 'val', 'test']:
    imgs = {os.path.splitext(f)[0] for f in os.listdir(f'{MERGED}/{s}/images')}
    lbls = {os.path.splitext(f)[0] for f in os.listdir(f'{MERGED}/{s}/labels')}
    print(s, '| ảnh thiếu nhãn:', len(imgs - lbls), '| nhãn thiếu ảnh:', len(lbls - imgs))
    for p in glob.glob(f'{MERGED}/{s}/labels/*.txt'):
        for line in open(p):
            parts = line.split()
            if parts and not (0 <= int(parts[0]) <= 13):
                bad += 1
print('Dòng nhãn có class id sai:', bad)

## Khai báo tham số huấn luyện
**Lần train thứ 2 (v2):** bài toán đổi từ băng chuyền sang *gắp quân từ khay lên bàn cờ*. Trong khay quân nằm xoay **bất kỳ 0–360°**, nên phải tăng xoay ảnh khi train từ 10° lên 180°.

Không train lại từ đầu mà **fine-tune từ `best.pt` của lần trước** (đã học tốt hình dạng quân, chỉ cần học thêm quân xoay), nên ít epoch hơn. Kết quả lưu vào run mới `yolo11n_640_rot`, không ghi đè run cũ để còn so sánh.

In [ ]:
from ultralytics import YOLO

DATA     = f'{MERGED}/data.yaml'
PROJECT  = f'{BASE}/runs'
NAME     = 'yolo11n_640_rot'
OLD_BEST = f'{PROJECT}/yolo11n_640/weights/best.pt'

EPOCHS  = 60
IMGSZ   = 640
BATCH   = 32

## Nạp mô hình từ lần train trước
Dùng `best.pt` của run `yolo11n_640` (đã huấn luyện trên dataset hợp nhất 14 lớp) làm điểm xuất phát.

In [ ]:
model = YOLO(OLD_BEST)
print(model.names)

## Huấn luyện mô hình (fine-tune)
Thay đổi so với lần 1:
- `degrees=180`: **xoay ngẫu nhiên tới ±180°**, vì quân trong khay nằm theo mọi hướng (lần 1 chỉ 10°).
- `fliplr=0.0`, `flipud=0.0`: vẫn tắt lật ảnh, vì lật làm chữ Hán bị đảo ngược (thành ảnh gương), gây nhầm lớp. Xoay 180° thì không đảo chữ nên an toàn.
- `optimizer='AdamW'`, `lr0=0.001`: learning rate nhỏ, phù hợp fine-tune, tránh làm hỏng những gì model đã học.
- `epochs=60`, `patience=20`: ít hơn lần 1 vì xuất phát từ model đã tốt.

Giữ nguyên: `save_period=10`, `close_mosaic=10`. Thời gian ước tính khoảng 40–60 phút trên T4.

In [ ]:
results = model.train(
    data=DATA,
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    patience=20,
    project=PROJECT,
    name=NAME,
    exist_ok=True,
    save_period=10,
    workers=2,
    optimizer='AdamW',
    lr0=0.001,
    fliplr=0.0,
    flipud=0.0,
    degrees=180,
    close_mosaic=10,
    plots=True,
)

## Tiếp tục huấn luyện nếu bị ngắt giữa chừng
Chỉ chạy ô này khi Colab ngắt kết nối lúc đang train. Chạy lại phần chuẩn bị môi trường, giải nén dataset và ô khai báo tham số, **bỏ qua** ô nạp mô hình và ô train, rồi chạy ô dưới để train tiếp từ `last.pt`.

In [ ]:
# model = YOLO(f'{PROJECT}/{NAME}/weights/last.pt')
# model.train(resume=True)

## Xem kết quả huấn luyện
Đọc `results.csv` và vẽ đường loss cùng mAP theo từng epoch. Loss giảm đều và mAP tăng rồi đi ngang là dấu hiệu huấn luyện tốt. Nếu loss val tăng trong khi loss train giảm thì mô hình đang overfit.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv(f'{PROJECT}/{NAME}/results.csv')
df.columns = df.columns.str.strip()

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
for c in ['train/box_loss', 'val/box_loss', 'train/cls_loss', 'val/cls_loss']:
    ax[0].plot(df['epoch'], df[c], label=c)
ax[0].set_title('Loss'); ax[0].set_xlabel('epoch'); ax[0].legend()

for c in ['metrics/mAP50(B)', 'metrics/mAP50-95(B)']:
    ax[1].plot(df['epoch'], df[c], label=c)
ax[1].set_title('mAP (val)'); ax[1].set_xlabel('epoch'); ax[1].legend()
plt.show()

print(df[['epoch', 'metrics/precision(B)', 'metrics/recall(B)',
          'metrics/mAP50(B)', 'metrics/mAP50-95(B)']].tail(3))

## Đánh giá trên tập test
Tập test chưa từng được dùng khi huấn luyện hay chọn mô hình, nên đây là con số khách quan để báo cáo. Mục tiêu trong đề cương là **mAP@0.5 ≥ 95%**. Bảng bên dưới cho biết từng lớp để phát hiện lớp nào còn yếu.

In [ ]:
best = YOLO(f'{PROJECT}/{NAME}/weights/best.pt')
metrics = best.val(data=DATA, split='test', imgsz=IMGSZ, project=PROJECT, name=f'{NAME}_test', exist_ok=True)

print(f'mAP@0.5     : {metrics.box.map50:.4f}')
print(f'mAP@0.5:0.95: {metrics.box.map:.4f}')
print(f'Precision   : {metrics.box.mp:.4f} | Recall: {metrics.box.mr:.4f}')

print('\nmAP@0.5 theo lớp:')
for i, name in best.names.items():
    print(f'  {name:15s} {metrics.box.ap50[i]:.4f}')

## Dự đoán thử trên ảnh test
Chọn ngẫu nhiên vài ảnh từ tập test để xem mô hình khoanh vùng và gán nhãn có hợp lý không. Bước này hữu ích để đưa vào báo cáo.

In [ ]:
import random
from IPython.display import Image, display

samples = random.sample(glob.glob(f'{MERGED}/test/images/*'), 4)
preds = best.predict(samples, imgsz=IMGSZ, conf=0.5, save=True,
                     project=PROJECT, name=f'{NAME}_pred', exist_ok=True)

for r in preds:
    display(Image(filename=os.path.join(r.save_dir, os.path.basename(r.path))))

## So sánh model cũ và mới trên tập test **đã xoay**
Tạo thêm 3 bản của tập test, xoay 90°, 180°, 270° (nhãn được xoay chính xác theo). Đánh giá cả model cũ (`degrees=10`) và model mới (`degrees=180`) trên 4 tập (0°, 90°, 180°, 270°).

Bảng này cho thấy model mới **không bị giảm độ chính xác khi quân xoay**, dùng để đưa vào báo cáo giải thích lý do train lại.

In [ ]:
import yaml, pandas as pd
import cv2

ROT = {90: cv2.ROTATE_90_CLOCKWISE, 180: cv2.ROTATE_180, 270: cv2.ROTATE_90_COUNTERCLOCKWISE}

def rot_label(x, y, w, h, deg):
    # toạ độ YOLO chuẩn hoá, xoay theo chiều kim đồng hồ
    if deg == 90:
        return 1 - y, x, h, w
    if deg == 180:
        return 1 - x, 1 - y, w, h
    return y, 1 - x, h, w          # 270

ROT_ROOT = '/content/test_rot'
yamls = {0: DATA}
for deg, code in ROT.items():
    root = f'{ROT_ROOT}/r{deg}'
    if not os.path.isdir(f'{root}/images'):
        os.makedirs(f'{root}/images'); os.makedirs(f'{root}/labels')
        for f in os.listdir(f'{MERGED}/test/images'):
            img = cv2.imread(f'{MERGED}/test/images/{f}')
            if img is None:
                continue
            cv2.imwrite(f'{root}/images/{f}', cv2.rotate(img, code))
            stem = os.path.splitext(f)[0]
            out = []
            for line in open(f'{MERGED}/test/labels/{stem}.txt'):
                p = line.split()
                if len(p) == 5:
                    x, y, w, h = rot_label(*map(float, p[1:]), deg)
                    out.append(f'{p[0]} {x:.6f} {y:.6f} {w:.6f} {h:.6f}')
            open(f'{root}/labels/{stem}.txt', 'w').write('\n'.join(out) + '\n')
    yp = f'{root}/data.yaml'
    with open(yp, 'w') as fh:
        yaml.safe_dump({'path': root, 'train': 'images', 'val': 'images', 'test': 'images',
                        'nc': 14, 'names': list(best.names.values())}, fh, sort_keys=False)
    yamls[deg] = yp

rows = []
for tag, w in [('cũ (degrees=10)', OLD_BEST), ('mới (degrees=180)', f'{PROJECT}/{NAME}/weights/best.pt')]:
    m = YOLO(w)
    row = {'model': tag}
    for deg, yp in yamls.items():
        r = m.val(data=yp, split='test', imgsz=IMGSZ, verbose=False, plots=False,
                  project=PROJECT, name=f'{NAME}_cmp', exist_ok=True)
        row[f'mAP50 {deg}°'] = round(r.box.map50, 4)
    rows.append(row)
print(pd.DataFrame(rows).to_string(index=False))

## Xuất mô hình sang ONNX
`best.pt` dùng được ngay với Ultralytics, còn ONNX là định dạng trung gian để chuyển sang TensorRT. Xuất với **`opset=12`** vì TensorRT 8.x trên Jetson Nano (JetPack 4.x) không đọc được opset mới hơn. **Lưu ý:** file TensorRT (`.engine`) phải được tạo trực tiếp trên Jetson của robot, không tạo trên Colab được, vì nó phụ thuộc vào GPU và phiên bản TensorRT của thiết bị.

In [ ]:
onnx_path = best.export(format='onnx', imgsz=IMGSZ, opset=12, simplify=True)
print('Đã xuất:', onnx_path)
print('Các file trong thư mục weights:', os.listdir(f'{PROJECT}/{NAME}/weights'))

## Tải kết quả về máy
Nén thư mục kết quả và tải về, giải nén vào `runs/yolo11n_640_rot/` trong dự án. Bản gốc vẫn còn trên Drive. Các file cần mang sang robot: `weights/best.pt` và `weights/best.onnx`.

In [ ]:
from google.colab import files

shutil.make_archive(f'/content/{NAME}', 'zip', f'{PROJECT}/{NAME}')
files.download(f'/content/{NAME}.zip')